In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [7]:
# Load the dataset
df = pd.read_csv('../data/globalAirQuality.csv')
print('Dataset shape:', df.shape)
print('Columns:', list(df.columns))
df.head()

Dataset shape: (18000, 15)
Columns: ['timestamp', 'country', 'city', 'latitude', 'longitude', 'pm25', 'pm10', 'no2', 'so2', 'o3', 'co', 'aqi', 'temperature', 'humidity', 'wind_speed']


,timestamp,country,city,latitude,longitude,pm25,pm10,no2,so2,o3,co,aqi,temperature,humidity,wind_speed
0,2025-11-04 18:25:17.554219,US,New York,40.713,-74.006,50.295,108.938,27.998,6.539,52.568,1.096,108,18.504,70.168,3.725
1,2025-11-04 19:25:17.554219,US,New York,40.713,-74.006,32.083,63.043,36.120,4.021,43.536,1.075,90,5.838,80.088,8.969
2,2025-11-04 20:25:17.554219,US,New York,40.713,-74.006,42.250,82.553,26.935,9.538,23.320,0.977,84,31.833,62.783,9.650
3,2025-11-04 21:25:17.554219,US,New York,40.713,-74.006,30.403,79.951,63.536,7.609,31.369,0.230,158,23.140,89.153,8.956
4,2025-11-04 22:25:17.554219,US,New York,40.713,-74.006,21.083,66.423,38.997,6.919,45.615,1.085,97,13.632,76.499,4.017


In [8]:
# Basic statistics
df.describe()

,latitude,longitude,pm25,pm10,no2,so2,o3,co,aqi,temperature,humidity,wind_speed
count,18000.000000,18000.000000,18000.000000,18000.000000,18000.000000,18000.000000,18000.000000,18000.000000,18000.000000,18000.000000,18000.000000,18000.000000
mean,23.065980,37.655560,40.369131,70.152228,32.055176,6.035508,48.065100,0.800595,104.645556,21.510251,57.714351,5.283910
std,26.156536,78.600701,17.647450,24.999440,13.820680,2.454790,14.950849,0.250254,25.616070,9.509444,18.844908,2.741712
min,-37.814000,-123.121000,0.025000,0.061000,0.013000,0.003000,0.114000,0.000000,16.000000,5.000000,25.002000,0.500000
25%,12.972000,2.352000,27.904500,53.125500,22.362500,4.360750,38.028500,0.633000,87.000000,13.357750,41.320000,2.937000
50%,29.232000,42.146000,40.286500,69.961000,32.019500,6.026000,48.142000,0.800500,103.000000,21.455500,57.847000,5.297000
75%,41.008000,103.820000,52.436250,87.256500,41.364250,7.715250,58.258500,0.969000,121.000000,29.688250,74.234750,7.662000
max,60.170000,174.763000,115.683000,161.810000,90.019000,16.559000,103.016000,1.832000,231.000000,37.998000,89.997000,9.999000


In [ ]:
# Check for missing values
df.isnull().sum()

In [ ]:
# Data types
df.dtypes

In [9]:
# Clean data - drop rows with missing aqi
df = df.dropna(subset=['aqi'])
print('After cleaning shape:', df.shape)

After cleaning shape: (18000, 15)


In [12]:
# EDA: Distribution of AQI Value
fig = px.histogram(df, x='aqi', nbins=50, title='AQI Value Distribution')
fig.update_layout(showlegend=False)
fig.write_image('../viz/aqi_distribution.png')
fig.show()

In [25]:
# EDA: AQI by Country (top 20)
top_countries = df['country'].value_counts().head(20).index
df_top = df[df['country'].isin(top_countries)]
fig = px.box(df_top, x='country', y='aqi', title='AQI Value by Country (Top 20)')
fig.write_image('../viz/aqi_by_country.png')
fig.show()

In [24]:
# EDA: Correlation heatmap
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr()
fig = px.imshow(corr_matrix, text_auto=True, title='Correlation Heatmap')
fig.write_image('../viz/correlation_heatmap.png')
fig.show()

In [23]:
# EDA: PM2.5 vs AQI
fig = px.scatter(df, x='pm25', y='aqi', title='PM2.5 vs AQI')
fig.write_image('../viz/pm25_vs_aqi.png')
fig.show()

In [17]:
# Preprocessing
# Encode categorical variables
le_country = LabelEncoder()
df['country_encoded'] = le_country.fit_transform(df['country'])

le_city = LabelEncoder()
df['city_encoded'] = le_city.fit_transform(df['city'])

# Select features (pollutants and weather data)
features = ['pm25', 'pm10', 'no2', 'so2', 'o3', 'co', 'temperature', 'humidity', 'wind_speed', 'country_encoded', 'city_encoded']
X = df[features]
y = df['aqi']

# Drop rows with NaN in features
X = X.dropna()
y = y.loc[X.index]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [18]:
# Train Random Forest model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

# Predictions
y_pred = rf_model.predict(X_test_scaled)

# Evaluation
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f'MAE: {mae:.2f}')
print(f'RMSE: {rmse:.2f}')
print(f'R²: {r2:.4f}')

# Actual vs Predicted plot
fig = px.scatter(x=y_test, y=y_pred, title='Actual vs Predicted AQI')
fig.add_trace(go.Scatter(x=[y_test.min(), y_test.max()], y=[y_test.min(), y_test.max()], mode='lines', name='Perfect Prediction'))
fig.update_layout(xaxis_title='Actual AQI', yaxis_title='Predicted AQI')
fig.write_image('../viz/actual_vs_predicted.png')

MAE: 0.08
RMSE: 0.56
R²: 0.9995


In [19]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

fig = px.bar(feature_importance, x='importance', y='feature', orientation='h', title='Feature Importance')
fig.write_image('../viz/feature_importance.png')

In [20]:
# Create a pipeline for the model
from sklearn.pipeline import Pipeline

# Define the pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

# Fit the pipeline
pipeline.fit(X_train, y_train)

# Example prediction
# Create a sample air quality data
sample_aq = pd.DataFrame({
    'pm25': [50],
    'pm10': [80],
    'no2': [25],
    'so2': [5],
    'o3': [40],
    'co': [0.8],
    'temperature': [25],
    'humidity': [60],
    'wind_speed': [5],
    'country_encoded': [le_country.transform(['US'])[0]],
    'city_encoded': [le_city.transform(['New York'])[0]]
})

# Make prediction
prediction = pipeline.predict(sample_aq)

print('Sample Air Quality Prediction:')
print(f'Predicted AQI: {prediction[0]:.2f}')

Sample Air Quality Prediction:
Predicted AQI: 99.26


In [22]:
# Save the pipeline
import joblib
joblib.dump(pipeline, '../models/air_quality_pipeline.joblib')
print('Pipeline saved to ../models/air_quality_pipeline.joblib')

Pipeline saved to ../models/air_quality_pipeline.joblib
